In [1]:
import sys

!{sys.executable} -m pip install \
langchain-community \
langchain-text-splitters \
langchain-groq \
langchain-huggingface \
sentence-transformers \
pypdf \
scikit-learn

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from sklearn.metrics.pairwise import cosine_similarity

from langchain_groq import ChatGroq

import numpy as np


pdf_path = "/Users/Aryaanil/Documents/survey cdc.pdf"


loader = PyPDFLoader(pdf_path)

documents = loader.load()


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)


texts = [doc.page_content for doc in chunks]


embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


document_embeddings = embedding_model.embed_documents(texts)

document_embeddings = np.array(document_embeddings)


groq_api_key = "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"


llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.3-70b-versatile"
)


questions = [

    "What is the main topic discussed in the PDF?",
    "Which libraries are used in the notes?",
    "Explain one concept mentioned in the document.",

    "Who is the president of France?",
    "What is the capital of Japan?"
]


def retrieve_documents(query, k=3):

    query_embedding = embedding_model.embed_query(query)

    similarity_scores = cosine_similarity(
        [query_embedding],
        document_embeddings
    )[0]

    top_indices = similarity_scores.argsort()[::-1][:k]

    return [texts[i] for i in top_indices]


for question in questions:

    retrieved_docs = retrieve_documents(question)

    context = "\n".join(retrieved_docs)


    prompt = f"""
    You are a strict RAG assistant.

    Rules:
    1. Answer ONLY from the provided context.
    2. If the answer is not clearly present in the context,
       respond exactly with:
       "I could not find this in the document."
    3. Do not use outside knowledge.
    4. Do not guess or hallucinate.

    Context:
    {context}

    Question:
    {question}
    """


    response = llm.invoke(prompt)

    print("\nQUESTION:")
    print(question)

    print("\nANSWER:")
    print(response.content)

    print("\n" + "=" * 60)

/var/folders/2q/c34xq8nn70lgm5zqqcds5x7m0000gn/T/ipykernel_23428/42731401.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


QUESTION:
What is the main topic discussed in the PDF?

ANSWER:
The main topic discussed is academic and student-related matters, with a focus on the student community, specifically happiness and life satisfaction.


QUESTION:
Which libraries are used in the notes?

ANSWER:
I could not find this in the document.


QUESTION:
Explain one concept mentioned in the document.

ANSWER:
One concept mentioned in the document is "happiness" which is discussed in Section B: Happiness & Emotional Well-being, where respondents are asked to rate their happiness in daily life on a scale of 1-10.


QUESTION:
Who is the president of France?

ANSWER:
I could not find this in the document.


QUESTION:
What is the capital of Japan?

ANSWER:
I could not find this in the document.

